# Lesson 1 - Semantic Search

Welcome to Lesson 1. 

To access the `requirement.txt` file, go to `File` and click on `Open`.
 
I hope you enjoy this course!

### Import the Needed Packages

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from DLAIUtils import Utils
import DLAIUtils

import os
import time
import torch

In [4]:
from tqdm.auto import tqdm

### Load the Dataset

In [6]:
# dataset = load_dataset('sentence-transformers/quora-duplicates', "pair-class", split='train[240000:290000]')
dataset = load_dataset('quora', split='train[240000:290000]')

Generating train split: 100%|██████████| 404290/404290 [00:34<00:00, 11852.68 examples/s]


In [7]:
dataset[:5]

{'questions': [{'id': [207550, 351729],
   'text': ['What is the truth of life?', "What's the evil truth of life?"]},
  {'id': [33183, 351730],
   'text': ['Which is the best smartphone under 20K in India?',
    'Which is the best smartphone with in 20k in India?']},
  {'id': [351731, 351732],
   'text': ['Steps taken by Canadian government to improve literacy rate?',
    'Can I send homemade herbal hair oil from India to US via postal or private courier services?']},
  {'id': [37799, 94186],
   'text': ['What is a good way to lose 30 pounds in 2 months?',
    'What can I do to lose 30 pounds in 2 months?']},
  {'id': [351733, 351734],
   'text': ['Which of the following most accurately describes the translation of the graph y = (x+3)^2 -2 to the graph of y = (x -2)^2 +2?',
    'How do you graph x + 2y = -2?']}],
 'is_duplicate': [False, True, False, True, False]}

In [8]:
questions = []
for record in dataset['questions']:
    questions.extend(record['text'])
question = list(set(questions))
print('\n'.join(question[:10]))
print('-' * 50)
print(f'Number of questions: {len(question)}')

If you could live anywhere in the world where would you live and why? Pics appreciated!
How can a middle class student in India afford MS in USA?
Why am I always attracted to guys who don't like me?
How do I decorate a room with an 80s theme?
What is a good wholesale forum?
Are men superior to women? If so, in what ways?
Are all religions flawed?
Are there any conflict minerals present in floppy disks or floppy disk drives?
How do I reset my Gmail account password?
Why don't many people posting questions on Quora check Google first?
--------------------------------------------------
Number of questions: 88919


### Check cuda and Setup the model

**Note**: "Checking cuda" refers to checking if you have access to GPUs (faster compute). In this course, we are using CPUs. So, you might notice some code cells taking a little longer to run.

We are using *all-MiniLM-L6-v2* sentence-transformers model that maps sentences to a 384 dimensional dense vector space.

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    print('Sorry no cuda.')
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

Sorry no cuda.


In [10]:
query = 'which city is the most populated in the world?'
xq = model.encode(query)
xq.shape

(384,)

### Setup Pinecone

In [17]:
utils = Utils()
PINECONE_API_KEY = utils.get_pinecone_api_key()
print(PINECONE_API_KEY)

pcsk_3VVpc7_3ZFXrZjD715cGerfVHyPmW7uNkNHL9GS7Pm95D2ZrJi6d3xrbyUSZACNXBZcrfJ


In [20]:
pinecone = Pinecone(api_key=PINECONE_API_KEY)
INDEX_NAME = utils.create_dlai_index_name('dl-ai')

if INDEX_NAME in [index.name for index in pinecone.list_indexes()]:
    pinecone.delete_index(INDEX_NAME)
print(INDEX_NAME)
pinecone.create_index(name=INDEX_NAME, 
    dimension=model.get_sentence_embedding_dimension(), 
    metric='cosine',
    spec=ServerlessSpec(cloud='aws', region='us-east-1'))

index = pinecone.Index(INDEX_NAME)
print(index)

dl-ai-xsow2glswtf3oapaiuwdjxh9zytekynm6yya


### Create Embeddings and Upsert to Pinecone

In [23]:
batch_size=200
vector_limit=10000

questions = question[:vector_limit]

import json

for i in tqdm(range(0, len(questions), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(questions))
    # create IDs batch
    ids = [str(x) for x in range(i, i_end)]
    # create metadata batch
    metadatas = [{'text': text} for text in questions[i:i_end]]
    # create embeddings
    xc = model.encode(questions[i:i_end])
    # create records list for upsert
    records = zip(ids, xc, metadatas)
    # upsert to Pinecone
    index.upsert(vectors=records)

100%|██████████| 50/50 [02:41<00:00,  3.23s/it]


In [24]:
index.describe_index_stats()

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 10000}},
 'total_vector_count': 10000}

### Run Your Query

In [25]:
# small helper function so we can repeat queries later
def run_query(query):
  embedding = model.encode(query).tolist()
  results = index.query(top_k=10, vector=embedding, include_metadata=True, include_values=False)
  for result in results['matches']:
    print(f"{round(result['score'], 2)}: {result['metadata']['text']}")

In [26]:
run_query('which city has the highest population in the world?')

0.7: Where is the most beautiful city in the world?
0.61: Which city deserves to be the Capital of the World?
0.58: What country has the most beautiful people?
0.58: What is the biggest country?
0.54: What is the most remote place in the world?
0.54: Which country is the world's largest democracy?
0.53: Which is the biggest arena in the world?
0.51: What do you think is the greatest country in the world?
0.5: What are the best cities in the USA?
0.5: What are the best cities to visit in Europe?


In [27]:
query = 'how do i make chocolate cake?'
run_query(query)

0.64: How do you bake a 10" cake?
0.55: How can I learn about baking cakes and desserts?
0.55: What is the difference between chocolate and truffles and how are they made?
0.54: How do you make a crispy batter?
0.51: How do you make whipped cream like Starbucks makes it?
0.49: How is pumpkin pie made?
0.48: Where can I get an unique taste for cupcakes in Gold Coast?
0.47: What is the recipe for a spinach artichoke dip from the Cheesecake Factory?
0.46: What are some really simple recipes?
0.46: Where can I get very nice and original flavor cupcakes in Gold Coast?
